# micrograd 5: is my engine right, and why does it need tanh?

Karpathy, *Zero to Hero* #1, ~1:39–1:43 (the PyTorch section you skipped on Saturday), then two experiments of our own.

Goal: (1) **prove** your gradients are right against an independent engine, (2) see why a falling loss is *not* that proof, (3) watch a no-tanh network hit a floor you calculated in advance, (4) write the README yourself.

**Rules:** predict before you run, in the comment provided. A wrong prediction you can explain beats a right guess.

## Block 0: cold recall (paper, 10 min)
Answers go in the chat, not here. No notes.
1. Why does `backward()` run the closures in reverse topological order? Use the word **final**.
2. y = w₂(w₁x + b₁) + b₂: multiply it out. Why can't it fit y = x² however long you train? What would 10 such layers give?
3. `w.grad = −2`. If w goes up by 0.01, what happens to the loss? Does `w.data += -0.05 * w.grad` move w up or down, and by how much?

## Carry-over
`Value`, plus `Neuron` / `Layer` / `MLP` as you built them on Saturday. Run it. (You'll change `Neuron` in Block 3.)

In [ ]:
import math, random
import torch
from viz import draw_dot


class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only int/float powers"
        out = Value(self.data ** other, (self,), f'**{other}')
        def _backward():
            self.grad += other * self.data ** (other - 1) * out.grad
        out._backward = _backward
        return out

    def exp(self):
        out = Value(math.exp(self.data), (self,), 'exp')
        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), 'tanh')
        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out

    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other)
    def __truediv__(self, other): return self * other**-1
    def __radd__(self, other): return self + other
    def __rmul__(self, other): return self * other
    def __rsub__(self, other): return other + (-self)

    def backward(self):
        order, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                order.append(v)
        build(self)
        self.grad = 1.0
        for node in reversed(order):
            node._backward()


class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))

    def __call__(self, x):
        act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
        return act.tanh()

    def parameters(self):
        return self.w + [self.b]


class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        params = []
        for n in self.neurons:
            params.extend(n.parameters())
        return params


class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]


xs = [[2.0, 3.0, -1.0], [3.0, -1.0, 0.5], [0.5, 1.0, 1.0], [1.0, 1.0, -1.0]]
ys = [1.0, -1.0, -1.0, 1.0]
print("ready")

## Block 1: the same neuron in PyTorch (~1:39–1:43, 20 min)
Watch 1:39–1:43 first, then come back. This is the micrograd 2 neuron: x1 = 2, x2 = 0, w1 = −3, w2 = 1, b = 6.8813…

1. **Before running:** write `o` and all five gradients in the comment. You computed these by hand in micrograd 2. If you can't recall them, *derive* them: start from o.grad = 1, go through tanh (1 − o²), then through `+` and `*`.
2. `requires_grad`: PyTorch's default for a new tensor is **False**. Why would a library default to *not* tracking gradients? Of these five, which would you never need a gradient for when training, and why does Karpathy turn it on anyway?
3. `.item()`: what does it give you that `o.data` doesn't?
4. In one sentence: what does PyTorch store in one node that micrograd stores in many? (Hint: what is a *tensor*?)

In [ ]:
x1 = torch.tensor([2.0]).double();                 x1.requires_grad = True
x2 = torch.tensor([0.0]).double();                 x2.requires_grad = True
w1 = torch.tensor([-3.0]).double();                w1.requires_grad = True
w2 = torch.tensor([1.0]).double();                 w2.requires_grad = True
b  = torch.tensor([6.8813735870195432]).double();  b.requires_grad = True

n = x1*w1 + x2*w2 + b
o = torch.tanh(n)

# predict: o = ___ | x1 ___ | w1 ___ | x2 ___ | w2 ___ | b ___
o.backward()

print("o ", o.data.item())
for name, t in [("x1", x1), ("w1", w1), ("x2", x2), ("w2", w2), ("b", b)]:
    print(name, t.grad.item())

5. **Why `.double()`?** PyTorch's default is float32; Python's `float` is float64. In micrograd 1 you found float64 keeps about 16 significant figures. **Predict:** in float32, `o` matches the float64 value to about ___ significant figures. Then run.
6. If you were cross-checking gradients in float32, what size of disagreement would you have to *expect* and ignore? Why does that make a weaker test?

In [ ]:
# predict: float32 o agrees with float64 o to about ___ significant figures
n32 = torch.tensor(2.0)*torch.tensor(-3.0) + torch.tensor(0.0)*torch.tensor(1.0) + torch.tensor(6.8813735870195432)
o32 = torch.tanh(n32)
print(f"float64: {o.item():.17f}")
print(f"float32: {o32.item():.17f}")
print("dtype:", o32.dtype)

## Block 2: cross-check the whole MLP (20 min)
One neuron agreeing isn't much. Now all 41 gradients of a real loss.

The cell below builds a micrograd MLP (seeded, so you and I get the same one) and copies every weight into PyTorch tensors with the **same structure**: `tlayers[i][j]` is `(w, b)` for neuron j of layer i, where `w` is a tensor of that neuron's weights.

1. **Write `torch_forward`**, mirroring your `MLP.__call__`: for each layer, each neuron computes tanh(w · x + b); collect the outputs into the next `x`. Tools: `(w * x).sum()` is a dot product; `torch.stack(list_of_tensors)` turns a list into one tensor.
2. **Predict** the largest disagreement between your 41 gradients and PyTorch's: exactly 0? About 1e-3? About 1e-15? Why?

In [ ]:
random.seed(1)
mlp = MLP(3, [4, 4, 1])

def to_torch(mlp):
    return [[(torch.tensor([w.data for w in n.w], dtype=torch.float64, requires_grad=True),
              torch.tensor(n.b.data, dtype=torch.float64, requires_grad=True))
             for n in layer.neurons]
            for layer in mlp.layers]

def torch_forward(tlayers, x):
    # mirror MLP.__call__: layer by layer, neuron by neuron
    ___

def compare(mlp, tlayers):
    """Largest |micrograd grad - PyTorch grad| over every parameter, in the same order as mlp.parameters()."""
    tgrads = [g.item() for layer in tlayers for (w, b) in layer for g in list(w.grad) + [b.grad]]
    mgrads = [p.grad for p in mlp.parameters()]
    assert len(tgrads) == len(mgrads)
    return max(abs(m - t) for m, t in zip(mgrads, tgrads))

tlayers = to_torch(mlp)

# micrograd
loss = sum((mlp(x) - y)**2 for x, y in zip(xs, ys))
loss.backward()

# PyTorch
tloss = sum((torch_forward(tlayers, torch.tensor(x, dtype=torch.float64)) - y)**2 for x, y in zip(xs, ys)).sum()
tloss.backward()

# predict: max disagreement ≈ ___ because ___
print("loss   micrograd", loss.data, "  pytorch", tloss.item())
print("max grad disagreement:", compare(mlp, tlayers))

3. Why isn't it exactly 0? Run the next cell, then explain in one line. (Which order does each engine add things in?)

In [ ]:
print((0.1 + 0.2) + 0.3)
print(0.1 + (0.2 + 0.3))

4. **CS × Phil (2 min, in the chat):** two engines agree to 15 decimal places. Does that *prove* yours is correct? Name one way both could be wrong together. Then: the numerical check from micrograd 1 (nudge by h, measure) is a *third* witness. Why is it more independent of your engine than PyTorch is, even though it's less precise?

### The planted bug
The next cell swaps **one** method of `Value` for a version with a **one-character** change in its `_backward`. Don't decode it: just run it.

In [ ]:
import base64
exec(base64.b64decode("aW1wb3J0IG1hdGggYXMgX20KVmFsdWUuX3JlYWxfdGFuaCA9IFZhbHVlLnRhbmgKZGVmIF90KHNlbGYpOgogICAgdCA9IF9tLnRhbmgoc2VsZi5kYXRhKQogICAgb3V0ID0gVmFsdWUodCwgKHNlbGYsKSwgJ3RhbmgnKQogICAgZGVmIF9iYWNrd2FyZCgpOgogICAgICAgIHNlbGYuZ3JhZCArPSAoMSAtIHQpICogb3V0LmdyYWQKICAgIG91dC5fYmFja3dhcmQgPSBfYmFja3dhcmQKICAgIHJldHVybiBvdXQKVmFsdWUudGFuaCA9IF90Cg==").decode())
print("one character changed somewhere in Value")

5. **Predict, then run:** you train seed 1's network for 30 steps with the buggy engine. Does the loss still fall? Near 0, or stuck?

In [ ]:
def train(mlp, X, Y, steps, lr, every=5):
    for k in range(steps):
        ypred = [mlp(x) for x in X]
        loss = sum((yp - y)**2 for y, yp in zip(Y, ypred))
        for p in mlp.parameters():
            p.grad = 0.0
        loss.backward()
        for p in mlp.parameters():
            p.data += -lr * p.grad
        if k % every == 0 or k == steps - 1:
            print(k, round(loss.data, 5))
    return mlp

# predict: loss after 30 steps ≈ ___
random.seed(1)
buggy = train(MLP(3, [4, 4, 1]), xs, ys, steps=30, lr=0.05)

6. Now the cross-check on the buggy engine. Rebuild seed 1's network, run both engines, `compare`. What does it say?

In [ ]:
random.seed(1)
mlp = MLP(3, [4, 4, 1])
tlayers = to_torch(mlp)
loss = sum((mlp(x) - y)**2 for x, y in zip(xs, ys)); loss.backward()
tloss = sum((torch_forward(tlayers, torch.tensor(x, dtype=torch.float64)) - y)**2 for x, y in zip(xs, ys)).sum(); tloss.backward()
print("max grad disagreement:", compare(mlp, tlayers))

7. **Localise it.** The cross-check says *something* is wrong, not *what*. Test each operation on its own against PyTorch: this is what a unit test is. Which one fails? Then, without decoding the cell: what should that `_backward` multiply by, and what one-character slip would give you the wrong numbers you see? Check your guess against the numbers (x = 0.6).

In [ ]:
def check_op(name, f_micro, f_torch, a=0.6):
    x = Value(a); y = f_micro(x); y.backward()
    t = torch.tensor(a, dtype=torch.float64, requires_grad=True); u = f_torch(t); u.backward()
    print(f"{name:6s} value diff {abs(y.data - u.item()):.1e}   grad: micrograd {x.grad:.6f}  pytorch {t.grad.item():.6f}")

check_op("add",  lambda v: v + 3.0,   lambda t: t + 3.0)
check_op("mul",  lambda v: v * v,     lambda t: t * t)
check_op("pow",  lambda v: v ** 3,    lambda t: t ** 3)
check_op("exp",  lambda v: v.exp(),   lambda t: t.exp())
check_op("tanh", lambda v: v.tanh(),  lambda t: t.tanh())

8. **Why did training work at all with a wrong gradient?** Compare the buggy local derivative with the right one for t between −1 and 1: is the sign ever different? What does gradient descent actually *need* from a gradient to make the loss fall?
9. Now seeds 0 and 2 with the bug (cell below). Predict first. What does this tell you about using "the loss went down" as your test? (Compare micrograd 4's forgotten `zero_grad`.)

In [ ]:
# predict: seeds 0 and 2 with the bug end ___
for s in [0, 2]:
    print("seed", s)
    random.seed(s)
    train(MLP(3, [4, 4, 1]), xs, ys, steps=30, lr=0.05, every=10)

Reveal and restore (run both):

In [ ]:
print(base64.b64decode("aW1wb3J0IG1hdGggYXMgX20KVmFsdWUuX3JlYWxfdGFuaCA9IFZhbHVlLnRhbmgKZGVmIF90KHNlbGYpOgogICAgdCA9IF9tLnRhbmgoc2VsZi5kYXRhKQogICAgb3V0ID0gVmFsdWUodCwgKHNlbGYsKSwgJ3RhbmgnKQogICAgZGVmIF9iYWNrd2FyZCgpOgogICAgICAgIHNlbGYuZ3JhZCArPSAoMSAtIHQpICogb3V0LmdyYWQKICAgIG91dC5fYmFja3dhcmQgPSBfYmFja3dhcmQKICAgIHJldHVybiBvdXQKVmFsdWUudGFuaCA9IF90Cg==").decode())

In [ ]:
Value.tanh = Value._real_tanh
check_op("tanh", lambda v: v.tanh(), lambda t: t.tanh())   # should agree again

**Stretch (only if ahead): what PyTorch really does.** In `tlayers[0]` there are four `w` tensors of length 3. Stack them into one tensor `W`. **Predict its shape.** Then one whole layer is a single line: `torch.tanh(W @ x + bvec)`. Check it matches `torch_forward`'s first layer. This is how makemore will look: one node per *matrix*, not per number.

In [ ]:
# predict: W.shape = ___

## Break (5 min)

## Block 3: why tanh, by experiment (20 min)
Saturday's cold check missed this twice, so we're going to *see* it.

**Step 1: make the non-linearity optional.** Edit the carry-over classes (then re-run that cell):
- `Neuron(nin, nonlin=True)`: if `nonlin` is False, return `act` without tanh.
- `Layer(nin, nout, nonlin=True)`: pass it on to each neuron.
- `MLP(nin, nouts, nonlin=True)`: hidden layers use `nonlin`; the **last layer is always linear**.

Before coding, answer: with tanh on the output neuron, what's the largest number the net can *ever* output? Why does that matter if you want to fit y = x² out to x = 2?

**Step 2: data.** Seven points, x from −1 to 1, y = x².

**Predict before training the no-tanh net** (in the chat):
- (a) With `nonlin=False`, what kind of function is the whole network, however many layers?
- (b) What's the best such function through these 7 points? Its slope, and its intercept? (You've been caught assuming symmetry before. Here, say *why* the symmetry argument is valid.)
- (c) So what's the **lowest loss** it can ever reach? Work it out by hand, as a fraction.

In [ ]:
X = [[i/3] for i in range(-3, 4)]
Y = [x[0]**2 for x in X]
print([round(x[0], 3) for x in X])
print([round(y, 3) for y in Y])

# predict: best possible loss for a no-tanh net = ___ (as a fraction)
random.seed(0)
linear = train(MLP(1, [8, 8, 1], nonlin=False), X, Y, steps=300, lr=0.01, every=30)
print("predictions:", [round(linear(x).data, 3) for x in X])

- Did it stop at your number? Look at the predictions: what has it learnt?
- Is that a training failure (more steps, better learning rate would fix it) or a *model-class* failure? How do you know?

**Step 3: same network, with tanh.** Predict the final loss: above or below your floor?

In [ ]:
random.seed(0)
net = train(MLP(1, [8, 8, 1]), X, Y, steps=500, lr=0.02, every=50)

**Step 4: has it learnt the curve or stored seven dots?** It has (count them) ___ parameters for 7 data points.
- **Predict** its outputs at x = −5/6, −1/6, 1/6, 1/2, 5/6: points it never saw, *between* the training points.
- **Predict** its outputs at x = 1.5, 2, 3: points *outside* the range it saw. (Think about what each hidden tanh does when its input gets large.)

In [ ]:
# predict between: ___     predict outside: ___
for xt in [-5/6, -1/6, 1/6, 1/2, 5/6, 1.5, 2.0, 3.0]:
    print(f"x = {xt:6.3f}   true {xt**2:6.3f}   net {net([xt]).data:6.3f}")

**CS × Phil (in the chat, 3 min):**
- Between the points it generalises; outside them it fails. So has it learnt "y = x²", or "y = x² *on [−1, 1]*", or something else? What would you need to know to decide?
- Nothing in the 7 data points rules out a function that matches x² on [−1, 1] and does anything at all beyond. This is **Hume's problem of induction** in miniature. On Wednesday, Aaronson §7 is PAC learning: the CS attempt to say when learning from examples is *justified*. Write down one question you want §7 to answer.
- Your Aaronson §4 zip test said compression counts when it *generalises*. 97 parameters for 7 points isn't compression in size, yet it generalises between points. What is doing that work, if not storage?

## Block 4: README (20 min, from memory)
Open `micrograd/README.md`. **Without looking at the log or your earlier draft**, add under *Backprop in my own words*:

1. **Why tanh?** (3–4 sentences.) Use your numbers from Block 3: the no-tanh floor you predicted and hit, and the tanh net's loss.
2. **How I know my engine is right.** (3–4 sentences.) The PyTorch cross-check and its number; why a falling loss isn't evidence (the planted bug); what a cross-check *can't* prove.
3. Tick the PyTorch box.

Paste it in the chat for review **before** you push. Then:
`git add -A && git commit -m "micrograd 5: PyTorch cross-check, tanh experiment, README" && git push`

## Wrap (5 min)
For the activity log: what you **learned**, what **surprised** you, and the **question** it opened. One sentence each, your words.